<a href="https://colab.research.google.com/github/hisjeans/data-analysis-ml/blob/main/5_2%EA%B5%90%EC%B0%A8%EA%B2%80%EC%A6%9D%EA%B7%B8%EB%A6%AC%EB%93%9C%EC%84%9C%EC%B9%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 교차 검증과 그리드 서치

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/rickiepark/hg-mldl/blob/master/5-2.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

## 검증 세트

In [1]:
import pandas as pd

wine = pd.read_csv('https://bit.ly/wine-date')

In [2]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [3]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42)

In [34]:
print(train_input.shape, test_input.shape)

(5197, 3) (1300, 3)


In [35]:
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42)

In [36]:
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


## 노트북 전체 코드 설명

이 노트북은 **교차 검증(Cross-validation)**과 **그리드 서치(Grid Search)**, 그리고 **랜덤 서치(Random Search)**를 사용하여 머신러닝 모델의 성능을 향상시키는 과정을 보여줍니다. 의사결정 트리(Decision Tree) 모델을 예시로 사용합니다.

### 1. 검증 세트 (Validation Set)

이 섹션에서는 모델 학습을 위한 데이터를 준비하고, **훈련 세트(Training Set)**와 **검증 세트(Validation Set)**로 나누어 모델의 과대적합(Overfitting) 여부를 확인합니다.

*   `pandas`를 이용해 와인 데이터셋을 로드합니다.
*   데이터를 특성(features) `data`와 타겟(target) `target`으로 분리합니다.
*   `train_test_split`을 사용하여 전체 데이터를 훈련 세트와 테스트 세트로 나눕니다.
*   훈련 세트를 다시 `train_input` (훈련 세트)과 `val_input` (검증 세트)으로 분리합니다.
*   `DecisionTreeClassifier` 모델을 `sub_input`으로 훈련시키고, 훈련 세트와 검증 세트 각각에 대한 정확도를 출력하여 과대적합 경향을 확인합니다.

### 2. 교차 검증 (Cross-validation)

모델의 일반화 성능을 더 신뢰할 수 있게 평가하기 위해 교차 검증을 사용합니다.

*   `cross_validate` 함수를 사용하여 훈련 세트(`train_input`, `train_target`)에 대한 5-폴드 교차 검증을 수행합니다.
*   `scores` 딕셔너리에는 각 폴드에서의 훈련 시간, 평가 시간, 테스트 점수 등이 포함됩니다.
*   `np.mean(scores['test_score'])`를 통해 5개 폴드의 평균 테스트 점수를 계산합니다.
*   `StratifiedKFold`를 사용하여 타겟 클래스의 비율을 유지하면서 5-폴드 교차 검증을 수행하고, 다시 평균 점수를 계산합니다.
*   `n_splits=10`, `shuffle=True`, `random_state=42`를 설정하여 10-폴드 섞은(shuffled) 층화 교차 검증을 수행하고 평균 점수를 확인합니다. 이는 데이터 분할을 더욱 안정적으로 만듭니다.

### 3. 하이퍼파라미터 튜닝 (Hyperparameter Tuning) - 그리드 서치

모델의 성능을 최적화하기 위해 하이퍼파라미터를 조정하는 과정입니다. 여기서는 **그리드 서치(Grid Search)**를 사용합니다.

*   `GridSearchCV`를 사용하여 `min_impurity_decrease` (불순도 감소 최소량) 하이퍼파라미터에 대한 최적 값을 찾습니다.
*   `n_jobs=-1`은 모든 CPU 코어를 사용하여 병렬 처리를 하도록 설정합니다.
*   `gs.fit`을 통해 모델을 훈련시키고, `best_estimator_` (최적의 하이퍼파라미터로 훈련된 모델)와 `best_params_` (최적의 하이퍼파라미터 조합)를 얻습니다.
*   `gs.cv_results_['mean_test_score']`를 통해 각 하이퍼파라미터 조합에 대한 교차 검증 평균 점수를 확인합니다.
*   더 많은 하이퍼파라미터 (`max_depth`, `min_samples_split`)를 포함하는 `params` 딕셔너리를 정의하여 더 넓은 범위의 그리드 서치를 수행하고 최적의 조합과 최고 점수를 출력합니다.

### 4. 랜덤 서치 (Random Search)

그리드 서치가 모든 가능한 조합을 탐색하는 반면, 랜덤 서치는 주어진 분포에서 무작위로 샘플링하여 하이퍼파라미터를 탐색합니다. 이는 탐색 공간이 넓을 때 효율적일 수 있습니다.

*   `scipy.stats`의 `uniform` (연속 균등 분포)과 `randint` (정수 균등 분포)를 사용하여 하이퍼파라미터의 탐색 범위를 확률 분포로 정의합니다.
*   `RandomizedSearchCV`를 사용하여 정의된 분포에서 `n_iter` (샘플링할 조합 수)만큼 무작위로 하이퍼파라미터 조합을 샘플링하여 교차 검증을 수행합니다.
*   `gs.fit` 이후 `best_params_`와 `np.max(gs.cv_results_['mean_test_score'])`를 통해 최적의 하이퍼파라미터 조합과 최고 교차 검증 점수를 확인합니다.
*   최적의 모델 (`gs.best_estimator_`)을 사용하여 최종 `test_input`에 대한 모델의 성능을 평가합니다.

### 5. 확인 문제

마지막 섹션은 `DecisionTreeClassifier`의 `splitter` 매개변수를 'random'으로 설정했을 때의 성능 변화를 확인하는 문제입니다.

*   `splitter='random'`은 노드를 분할할 때 최적의 특성 대신 무작위로 특성을 선택하게 하여 모델의 편향(bias)을 줄이고 분산을 높일 수 있습니다.
*   앞서 정의한 `params`와 동일하게 `RandomizedSearchCV`를 수행하고, 최적의 하이퍼파라미터, 최고 평균 교차 검증 점수, 그리고 테스트 세트 점수를 출력합니다.

In [4]:
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42)

In [5]:
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [6]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


## 교차 검증

In [7]:
from sklearn.model_selection import cross_validate

scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.02293825, 0.0151155 , 0.01482916, 0.01360154, 0.01288462]), 'score_time': array([0.00206542, 0.00195622, 0.002033  , 0.00204158, 0.0020535 ]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [8]:
import numpy as np

print(np.mean(scores['test_score']))

0.855300214703487


In [9]:
from sklearn.model_selection import StratifiedKFold

scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.855300214703487


In [10]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


## 하이퍼파라미터 튜닝

In [11]:
from sklearn.model_selection import GridSearchCV

params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [12]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)

In [13]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [14]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [15]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [16]:
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [17]:
best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

{'min_impurity_decrease': 0.0001}


In [18]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
          'max_depth': range(5, 20, 1),#5~20 +1
          'min_samples_split': range(2, 100, 10)
          }

In [19]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42),\ params, n_jobs=-1) #멀티코어 사용
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [20]:
print(gs.best_params_)

{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}


In [21]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8683865773302731


### 랜덤 서치

In [22]:
from scipy.stats import uniform, randint

In [23]:
rgen = randint(0, 10)
rgen.rvs(10)

array([5, 7, 9, 3, 7, 5, 8, 5, 8, 2])

In [24]:
np.unique(rgen.rvs(1000), return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([ 96, 113,  98, 102,  95,  88, 100,  98, 106, 104]))

In [25]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.59701103, 0.28381542, 0.30408788, 0.57114159, 0.38493359,
       0.60243243, 0.53527067, 0.74580909, 0.28040685, 0.48592361])

In [26]:
params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25),
          }

In [27]:
from sklearn.model_selection import RandomizedSearchCV

gs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params,
                        n_iter=100, n_jobs=-1, random_state=42)
gs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c13af2260c0>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c13af2a9010>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c13af2a8050>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c13af2abb90>},
                   random_state=42)

In [28]:
print(gs.best_params_)

{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}


In [29]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8695428296438884


In [30]:
dt = gs.best_estimator_

print(dt.score(test_input, test_target))

0.86


## 확인문제

In [31]:
gs = RandomizedSearchCV(DecisionTreeClassifier(splitter='random', random_state=42), params,
                        n_iter=100, n_jobs=-1, random_state=42)
gs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42,
                                                    splitter='random'),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c13af2260c0>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c13af2a9010>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c13af2a8050>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c13af2abb90>},
                   random_state=42)

In [32]:
print(gs.best_params_)
print(np.max(gs.cv_results_['mean_test_score']))

dt = gs.best_estimator_
print(dt.score(test_input, test_target))

{'max_depth': 43, 'min_impurity_decrease': np.float64(0.00011407982271508446), 'min_samples_leaf': 19, 'min_samples_split': 18}
0.8458726956392981
0.786923076923077
